# The Capstone — One Failure, Four Services, Every Layer at Once

*A warm, step-by-step lab that ties the whole journey together. You bring the*
*protocols. We bring the graph that finally sees them as one thing.*

You have built four graphs by now — one for **OSPF**, one for **BGP**, one for
**MPLS**, one for **VXLAN**. Each taught you the same five words on a different
layer. This lab does the thing none of those four could do alone.

**The one idea:** a real outage does not respect protocol boundaries. One core link
drops and it ripples *upward* — the IGP loses a path, a BGP next-hop won't resolve,
an MPLS LSP breaks, a VXLAN leaf falls off the underlay — and **four different
services go down for four different-looking reasons**. Four on-call engineers open
four unrelated-looking tickets. A knowledge graph is the only place you can see all
four at once, and trace every one of them back to the *same* dead link.

By the end you will fail **one** core link and watch a single query return a list of
services spanning **BGP, MPLS, and VXLAN** — the whole blast radius of one cable.


## The bridge from the four courses

Everything you already know carries straight in. The five words are unchanged:

| Graph word | Plain meaning | You met it in |
|---|---|---|
| **node** | a thing | routers & areas (OSPF), prefixes (BGP), LSPs (MPLS), VNIs (VXLAN) |
| **edge** | a named, directed link | an adjacency, a session, a label binding, a VTEP tunnel |
| **label** | a node's *type* | `Router`, `Prefix`, `LSP`, `Leaf`, `Service` |
| **property** | a fact stored on a node or edge | `state='down'`, `criticality='critical'` |
| **traversal** | walking edges to answer a question | the blast radius — you've done it four times |

The genuinely **new** idea in this capstone is a single one: the **cross-layer
dependency edge** — the wire that says *"this BGP thing leans on that IGP thing."*
That one edge is what turns four separate graphs into a single stacked one. Learn to
draw it and you can model a failure cascading up through every protocol you run.

Everything shares one underlay. Break it, and count the fallout.


## First, the same reassurance as always: this is **not** machine learning

No training. No model weights. No embeddings. No AI guessing which service broke.
A knowledge graph is just **structured facts** — things, and the named links between
them — plus **queries** that walk those links. Everything here is **deterministic and
inspectable**: run it twice, get the same list; and for every service the graph flags,
it can show you the exact chain of edges from the dead link up to the revenue. If you
can read a routing table, you can read every line in this notebook.


## Setup — one import, one empty graph

**Theory (10 seconds).** `networkx` is a tiny Python library for building graphs in
memory; both it and `matplotlib` ship pre-installed in Colab, so there is nothing to
install. We use a **`DiGraph`** — a *directed* graph, where every edge is an arrow.
Direction is the whole game today: we will point every dependency arrow the same way,
**from the thing that leans, toward the thing it leans on**. Follow those arrows down
and you always end up at the underlay. Follow them *backwards* from a dead link and
you find everything that just fell over. That is the entire trick, stated once.

**One honest simplification up front.** A resilient core would have a second parallel
link, and `igp-reach` would depend on *either* one — so a single cable cut wouldn't
isolate anything. We collapse the core to a **single shared choke point on purpose**,
so the cross-layer cascade stands out cleanly instead of being buried under ECMP and
fast-reroute bookkeeping. The *shape* of the lesson is what generalizes to a real,
redundant core.


In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
from collections import deque

# A DiGraph is a *directed* graph: every edge is an arrow, from one node to another.
# Our convention: A --depends--> B means 'A needs B to work'.
G = nx.DiGraph()

print('Empty graph ready:', G)
print('Nodes:', G.number_of_nodes(), ' Edges:', G.number_of_edges())

**Checkpoint.** You should see `Empty graph ready` and `Nodes: 0  Edges: 0`. That
empty `G` is the whiteboard we will stack four protocol layers onto — bottom first.


## Step 1 — The shared underlay (the floor everything stands on)

**Theory.** At the very bottom of every design in your building sits a boring,
load-bearing fact: **two core routers, joined by a link, giving the IGP a path across
the core.** OSPF (or IS-IS) computes reachability over that path. Nothing exciting
lives here — which is exactly why nobody thinks about it until it fails and takes four
services with it.

We model three nodes, no more:

- **`core-r1`** and **`core-r2`** — two `Router` nodes.
- **`core-link`** — a `CoreLink` node carrying the one property that matters:
  `state`. We set it to **`down`** right now. This single node is the incident.
- **`igp-reach`** — an `IGPReachability` node: the summarized truth *"the IGP can get
  across the core."* It **`DEPENDS_ON`** the core link. When the link dies, this
  reachability dies with it.

Notice the direction: `igp-reach --DEPENDS_ON--> core-link`. The reachability leans on
the link. Keep that arrow pointing *down toward the failure* — every layer we add on
top will point the same way.


In [ ]:
# two core routers
G.add_node('core-r1', label='Router', role='core')
G.add_node('core-r2', label='Router', role='core')

# the one link whose loss is the whole story -- the failure is a property on the node
G.add_node('core-link', label='CoreLink', endpoints='core-r1<->core-r2', state='down')

# the summarized IGP truth that rides on that link
G.add_node('igp-reach', label='IGPReachability', scope='core')

# routers terminate the link; IGP reachability depends on it
G.add_edge('core-r1', 'core-link', rel='TERMINATES')
G.add_edge('core-r2', 'core-link', rel='TERMINATES')
G.add_edge('igp-reach', 'core-link', rel='DEPENDS_ON')   # <-- the load-bearing arrow

print('underlay nodes:', [n for n,d in G.nodes(data=True)])
print('core-link state:', G.nodes['core-link']['state'])

**Checkpoint.** You should see the four underlay nodes listed and `core-link state:
down`. That is the failure, sitting in the graph as a plain fact. Nothing depends on
it yet except the IGP — so let's stack the three protocol layers that secretly do.


## See the floor before we build on it

**Theory.** We'll draw as we go so each layer *lands* somewhere you can see. Nodes are
coloured by their **label**; the one node in `state='down'` is drawn **red**, because
that is the fact the whole graph will end up pointing at. This picture is generated
*from the data*, so it can never drift out of sync with the truth.


In [ ]:
PALETTE = {
    'Router': '#3aa0ff', 'CoreLink': '#5a6b7b', 'IGPReachability': '#0f7f74',
    'NextHop': '#7f8b3a', 'Prefix': '#9aa5ad', 'MPLSLSP': '#b0752f', 'VRF': '#c89b2a',
    'Leaf': '#8a6fb0', 'VNI': '#a86fb0', 'Service': '#c8702a',
}

def draw(G, title='Cross-protocol knowledge graph'):
    node_colors = []
    for n in G:
        d = G.nodes[n]
        if d.get('state') == 'down':
            node_colors.append('#d34b3f')          # the failure, in red
        else:
            node_colors.append(PALETTE.get(d.get('label'), '#cccccc'))
    pos = nx.spring_layout(G, seed=7, k=0.9)        # fixed seed => stable layout

    plt.figure(figsize=(11, 8))
    nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=1700, alpha=0.92)
    nx.draw_networkx_labels(G, pos, font_size=8)
    nx.draw_networkx_edges(G, pos, arrows=True, arrowsize=14, edge_color='#8894a0')
    nx.draw_networkx_edge_labels(G, pos, font_size=6,
        edge_labels={(u, v): d.get('rel', '') for u, v, d in G.edges(data=True)})
    plt.axis('off'); plt.title(title); plt.tight_layout(); plt.show()

draw(G, title='Step 1 — just the shared underlay')

**Reading the picture.** Two blue routers terminate a **red** `core-link` (down), and
the teal `igp-reach` node points at it with a `DEPENDS_ON` arrow. That's the floor.
Everything we add from here rides on `igp-reach` — and therefore, without knowing it,
on that red link.


## Step 2 — The BGP layer, and the **new idea**: a cross-layer edge

**Theory.** Here is the one genuinely new lesson of the whole capstone. A BGP route is
useless unless its **next-hop is reachable**, and *that reachability is resolved by the
IGP* — the famous BGP recursive lookup. So a BGP next-hop **leans on** an IGP fact.
In a graph you write that lean down literally, as an edge that crosses from the BGP
layer into the underlay layer:

> `next-hop --RESOLVED_BY--> igp-reach`

That is a **cross-layer dependency edge**. It looks exactly like every other edge —
same five words — but it is the wire that welds two protocols together. Add the rest of
the BGP story on top of it:

- a **`NextHop`** node (`192.0.2.10`) that is `RESOLVED_BY` `igp-reach`;
- a **`Prefix`** node (`10.10.10.0/24`) whose `NEXT_HOP_IS` that next-hop;
- a **`Service`** node, **`Checkout API`**, that `RUNS_ON` that prefix.

Every arrow points the same way — down toward the underlay. Trace `Checkout API` with
your finger: it runs on a prefix, whose next-hop is resolved by the IGP, which depends
on the core link. Four hops from revenue to a cable.


In [ ]:
# BGP rider -- note the cross-layer edge into the underlay
G.add_node('192.0.2.10',   label='NextHop')
G.add_node('10.10.10.0/24', label='Prefix')
G.add_node('Checkout API', label='Service', layer='BGP', criticality='critical')

G.add_edge('192.0.2.10',    'igp-reach',   rel='RESOLVED_BY')   # <-- CROSS-LAYER edge
G.add_edge('10.10.10.0/24', '192.0.2.10',  rel='NEXT_HOP_IS')
G.add_edge('Checkout API',  '10.10.10.0/24', rel='RUNS_ON')

print('BGP chain wired. Edges out of the next-hop:')
for u, v, d in G.out_edges('192.0.2.10', data=True):
    print(f'  {u} --{d["rel"]}--> {v}')

**Checkpoint.** You should see `192.0.2.10 --RESOLVED_BY--> igp-reach`. That one line
is the cross-layer edge — the BGP world now touches the underlay world. `Checkout API`
is quietly downstream of the red link, though nothing has *told* it yet. Two more
layers do the same trick.


## Step 3 — The MPLS layer (same shape, new protocol)

**Theory.** An MPLS **LSP** is a label-switched path that is *built over* an IGP path —
LDP or SR hands out labels for prefixes the IGP already knows how to reach. So an LSP
**leans on** the same underlay, through its own cross-layer edge:

> `LSP --BUILT_OVER--> igp-reach`

On top of it rides an L3VPN. We keep it to three nodes, mirroring the BGP shape:

- an **`LSP`** node (`LSP pe1->pe2`) that is `BUILT_OVER` `igp-reach`;
- a **`VRF`** node (`VRF-RED`) that is `TRANSPORTED_BY` that LSP;
- a **`Service`** node, **`Partner WAN`**, that `RUNS_IN` that VRF.

Different protocol, identical grammar. The customer's `Partner WAN` runs in a VRF,
transported by an LSP, built over the IGP, which depends on the core link. Same cable,
a completely different-looking failure.


In [ ]:
# MPLS rider -- its own cross-layer edge into the same underlay
# (label 'MPLSLSP' -- the unambiguous name; a bare 'LSP' is overloaded)
G.add_node('LSP pe1->pe2', label='MPLSLSP')
G.add_node('VRF-RED',      label='VRF')
G.add_node('Partner WAN',  label='Service', layer='MPLS', criticality='high')

G.add_edge('LSP pe1->pe2', 'igp-reach',    rel='BUILT_OVER')    # <-- CROSS-LAYER edge
G.add_edge('VRF-RED',      'LSP pe1->pe2', rel='TRANSPORTED_BY')
G.add_edge('Partner WAN',  'VRF-RED',      rel='RUNS_IN')

print('MPLS chain wired. Path from the service down to the underlay:')
for u, v, d in [('Partner WAN','VRF-RED','RUNS_IN'),
                ('VRF-RED','LSP pe1->pe2','TRANSPORTED_BY'),
                ('LSP pe1->pe2','igp-reach','BUILT_OVER')]:
    print(f'  {u} --{d}--> {v}')

**Checkpoint.** You should see the three-hop path `Partner WAN -> VRF-RED -> LSP -> ` 
ending at `igp-reach` via `BUILT_OVER`. Two services now hang off the same underlay
through two different protocols. One layer to go.


## Step 4 — The VXLAN layer (the overlay leans on the underlay too)

**Theory.** A VXLAN fabric is the purest statement of this whole idea: an **overlay
that is nothing without its underlay**. A leaf switch is a VTEP, and its tunnels to
other VTEPs only exist if the underlay can route between loopbacks. So the leaf's
underlay reachability **leans on** the same IGP path, through one more cross-layer edge:

> `leaf --UNDERLAY_VIA--> igp-reach`

The tenant story, again three nodes in the same shape:

- a **`Leaf`** node (`leaf-07`, a VTEP) that reaches the fabric `UNDERLAY_VIA`
  `igp-reach`;
- a **`VNI`** node (`VNI 50100`) that `RIDES_ON` that leaf;
- a **`Service`** node, **`Tenant-A App`**, that `RUNS_ON` that VNI.

Three protocols, three cross-layer edges (`RESOLVED_BY`, `BUILT_OVER`, `UNDERLAY_VIA`),
one shared `igp-reach`. Everything is now standing on the same red link.


In [ ]:
# VXLAN rider -- the third cross-layer edge into the shared underlay
G.add_node('leaf-07',   label='Leaf', role='VTEP')
G.add_node('VNI 50100', label='VNI', tenant='Tenant-A')
G.add_node('Tenant-A App', label='Service', layer='VXLAN', criticality='high')

G.add_edge('leaf-07',      'igp-reach', rel='UNDERLAY_VIA')     # <-- CROSS-LAYER edge
G.add_edge('VNI 50100',    'leaf-07',   rel='RIDES_ON')
G.add_edge('Tenant-A App', 'VNI 50100', rel='RUNS_ON')

cross = [(u, v) for u, v, d in G.edges(data=True)
         if d.get('rel') in ('RESOLVED_BY', 'BUILT_OVER', 'UNDERLAY_VIA')]
print('All cross-layer edges now feeding the shared underlay:')
for u, v in cross:
    print(f'  {u:14} --{G[u][v]["rel"]:13}--> {v}')

**Checkpoint.** You should see **three** cross-layer edges printed — `RESOLVED_BY`,
`BUILT_OVER`, `UNDERLAY_VIA` — all three pointing at `igp-reach`. Three protocols now
share one underlay. Time to see it, then break it.


## See the whole stack

**Theory.** This is the picture no single `show` command can draw: four protocol layers
in one frame, all narrowing down to a single red node. Look for the shape — a wide fan
of services at the top, funnelling through three cross-layer edges into one dead link
at the bottom.


In [ ]:
draw(G, title='One graph, four layers — everything funnels to the red core-link')

**Reading the picture.** Three orange `Service` nodes sit at the edges of the graph,
each on its own protocol chain, and every chain threads down through a cross-layer edge
into the teal `igp-reach`, which points at the **red** `core-link`. You can *see* that
one failure sits underneath all three services. Next we make the computer prove it.


## Step 5 — The 3 AM question: blast radius across every layer

**Theory.** Everything so far was setup for one query. Because every dependency arrow
points the *same way* — toward the underlay — the blast radius of a failed node is
simply: **everything that can reach it by following arrows.** So we start at the dead
`core-link` and walk **backwards** up every arrow (a reverse breadth-first search),
collecting every `Service` we pass. No per-protocol logic — the graph's shape does the
work. The query is also **conditioned on state**: if `core-link` is not `down`, it
returns nothing at all.

One failure in; a list of services from three protocols out.


In [ ]:
def impact(G, failed='core-link'):
    """Every Service that (transitively) depends on `failed`, if it is down."""
    if G.nodes[failed].get('state') != 'down':
        return []                              # healthy link => empty blast radius
    parent = {failed: None}                    # remember how we reached each node
    q = deque([failed])
    while q:
        n = q.popleft()
        for pred, _, d in G.in_edges(n, data=True):   # walk arrows BACKWARDS
            if pred not in parent:
                parent[pred] = n
                q.append(pred)
    hits = []
    for n in parent:
        if G.nodes[n].get('label') == 'Service':
            path, cur = [], n                  # rebuild the chain service -> ... -> failed
            while cur is not None:
                path.append(cur); cur = parent[cur]
            hits.append((G.nodes[n].get('layer'), n, path))
    return sorted(hits)

print('One core-link failure puts these services at risk:\n')
for layer, svc, path in impact(G):
    print(f'  [{layer:5}] {svc:14}  <-  ' + '  <-  '.join(path[1:]))

Stop and look at what just happened. **One** dead cable produced **three** services,
on **three** different protocols, from **one** query — and showed its work for each:
the BGP `Checkout API` because its next-hop won't resolve; the MPLS `Partner WAN`
because its LSP can't be built; the VXLAN `Tenant-A App` because its leaf fell off the
underlay. Three on-call engineers, three unrelated-looking tickets, **one** root cause
— and the graph is the only place all three were ever visible together. This is the
payoff the four earlier labs were building toward.


## Step 6 — What-if: repair the link, then break it again

**Theory.** The failure is a plain property on one node, so you can change it and
re-ask — running *"what if this fails?"* on a model, safely, with nobody paged. Flip
`core-link` to `up`: the entire multi-protocol blast radius clears at once. Flip it
back to `down`: all three services return together. Watch the whole cross-protocol set
appear and vanish from a single edit.


In [ ]:
names = lambda: sorted((layer, svc) for layer, svc, _ in impact(G))

G.nodes['core-link']['state'] = 'up'      # repair the cable
print('After repair:   ', names() or 'nothing at risk')

G.nodes['core-link']['state'] = 'down'    # break it again
print('After re-break: ', names())

**Checkpoint.** After the repair you should see `nothing at risk`; after re-breaking
it, all three come back: `[('BGP', 'Checkout API'), ('MPLS', 'Partner WAN'), ('VXLAN',
'Tenant-A App')]`. Same graph, same query — one property changed, and the answer across
three protocols responded. That is a model, not a drawing.


## Your turn #1 — a second service on the BGP prefix

Real prefixes rarely support just one thing. Suppose a **`Search API`** also runs on
`10.10.10.0/24`. Add it as a `Service` (layer `BGP`) and wire one `RUNS_ON` edge to
the prefix, then re-run the blast radius. A **fourth** service should surface from the
same dead link — because one edge of extra truth is all it takes, and you never touch
the query.

Fill in the `# TODO`, then run. The solution follows if you want to check.


In [ ]:
# TODO: add a 'Search API' Service node (layer='BGP', criticality='high'),
#       then a RUNS_ON edge from 'Search API' to '10.10.10.0/24'.

# G.add_node(...)
# G.add_edge(...)

print('Currently at risk:')
for layer, svc, _ in impact(G):
    print(f'  [{layer}] {svc}')

<details><summary><b>Solution — click to reveal</b></summary>

```python
G.add_node('Search API', label='Service', layer='BGP', criticality='high')
G.add_edge('Search API', '10.10.10.0/24', rel='RUNS_ON')

for layer, svc, _ in impact(G):
    print(f'[{layer}] {svc}')
```

Now `Search API` joins `Checkout API`, `Partner WAN`, and `Tenant-A App` — four
services, three protocols, one link. The blast radius grew the instant you told the
graph one more true thing. You didn't touch `impact` at all.
</details>


In [ ]:
# (Solution, applied -- so the rest of the notebook has this service in the graph.)
G.add_node('Search API', label='Service', layer='BGP', criticality='high')
G.add_edge('Search API', '10.10.10.0/24', rel='RUNS_ON')
print('At risk now:', sorted((l, s) for l, s, _ in impact(G)))

## Quiet reveal — you have been writing a *cross-protocol ontology*

Time to name it. Every step, you made two different kinds of decision:

- *"A `NextHop` is `RESOLVED_BY` an `IGPReachability`. An `LSP` is `BUILT_OVER` it. A
  `Leaf` reaches the fabric `UNDERLAY_VIA` it. A `Service` depends on a prefix / VRF /
  VNI."* — these are the **rules**: which node types exist and which edges are allowed
  to cross between layers. That is the **schema**, and its formal name is an
  **ontology**: the RFC of your graph, the agreed vocabulary written down as structure.
  The short edge names we used — `RESOLVED_BY`, `BUILT_OVER`, `UNDERLAY_VIA` — are
  **teaching stand-ins**, chosen to read cleanly in a diagram. This project's real
  interaction ontology spells the *same three dependencies* out in full:
  `BGP_ROUTE_DEPENDS_ON_NEXT_HOP_REACHABILITY` (in `ospf_bgp.yaml`),
  `MPLS_LSP_DEPENDS_ON_IGP_UNDERLAY` and `OVERLAY_DEPENDS_ON_UNDERLAY` (in
  `overlay_underlay.yaml`). Longer, but unambiguous — production vocabulary earns its
  verbosity. The idea is identical; only the spelling is friendlier here.

- *"This particular link is `core-link`, its state is `down`; this VNI is `50100`."* —
  these are the **instances**: the actual data.

The rule of thumb that keeps them straight forever: **if it has a hostname, an IP, a
VNI, or a serial number, it is data — never schema.** `Leaf` is schema; `leaf-07` is
data. `BUILT_OVER` is schema; "this LSP is built over the core IGP" is data. A small,
stable vocabulary of cross-layer edges; as many instances as your network has. That
discipline is what lets one graph span every protocol without becoming a swamp.


## A peek at the real thing (1/2) — the underlay and a cross-layer edge in Cypher

We used networkx for zero setup, but the *shapes* are exactly what you'd build in a
real graph database like **Neo4j**, whose language is **Cypher**. Cypher reads almost
like the arrows we've been drawing: `(node)-[:EDGE]->(node)`.

Here is the shared underlay plus the BGP cross-layer edge, in the **real production
form** — the actual labels and relationship names from this project's ontology, not our
friendly stand-ins. `MERGE` means *"find this or create it"* so re-running is safe; we
set properties with a separate `SET` (the re-run-safe idiom). This is **reference only**
— you don't run it here:

```cypher
MERGE (link:PhysicalLink {id: 'core-link'})
SET   link.state = 'down';                       // the failure, as a property

MERGE (igp:IGPReachability {id: 'igp-reach'})
MERGE (igp)-[:DEPENDS_ON]->(link);

MERGE (br:BGPRoute {id: '10.10.10.0/24'})
MERGE (br)-[r:BGP_ROUTE_DEPENDS_ON_NEXT_HOP_REACHABILITY]->(igp)
SET   r.note = 'BGP recursive next-hop lookup';  // the real cross-layer relationship
```

Line up the last statement with what you built: our toy
`G.add_edge('192.0.2.10', 'igp-reach', rel='RESOLVED_BY')` is the *same dependency* —
a BGP route that can't forward until the IGP resolves its next hop. The production
ontology just names it `BGP_ROUTE_DEPENDS_ON_NEXT_HOP_REACHABILITY` and hangs it on a
`BGPRoute`/`IGPReachability` pair. Same idea, grown-up spelling — and this is the form
that actually matches in the companion Neo4j lab.


## A peek at the real thing (2/2) — the whole blast radius in one Cypher line

Our `impact` walk is a dozen lines of Python. In Cypher the same cross-protocol
traversal is essentially **one** pattern, because a graph database lets you draw a
*variable-length* path — `*1..` means "one or more hops of these edge types" — and
finds every match itself. Here it is in the real production vocabulary, with the three
full cross-layer relationship names threaded into the path:

```cypher
MATCH (svc:BusinessService)
      -[:SUPPORTS|DEPENDS_ON
        |BGP_ROUTE_DEPENDS_ON_NEXT_HOP_REACHABILITY
        |MPLS_LSP_DEPENDS_ON_IGP_UNDERLAY
        |OVERLAY_DEPENDS_ON_UNDERLAY*1..]->
      (link:PhysicalLink {state: 'down'})
RETURN svc.layer AS layer, collect(DISTINCT svc.name) AS services_at_risk;
```

Read it as a sentence: *"match any BusinessService that reaches a down PhysicalLink
through one or more dependency edges of any layer."* That single pattern **is** the
reverse walk you wrote by hand — across BGP, MPLS, and VXLAN at once. Our notebook used
shorter edge names so the loop stayed readable; the engine and the vocabulary differ,
but the traversal is identical. If you can read the Python, you can read the Cypher.


## Your turn #2 — extend the blast radius onto a third protocol

So far the fourth service you added rode the BGP layer. Let's prove a *different*
protocol grows just as easily. A second tenant, **`Tenant-B App`**, rides a new
**`VNI 50200`** on the **same** `leaf-07` — so it inherits the leaf's `UNDERLAY_VIA`
dependency for free. Add two nodes and two edges, then re-run:

1. a `VNI` node `VNI 50200` with a `RIDES_ON` edge to `leaf-07`;
2. a `Service` node `Tenant-B App` (layer `VXLAN`) with a `RUNS_ON` edge to `VNI 50200`.

You never touch the underlay or the query — yet the one dead link now claims a fifth
service, on the VXLAN layer. Fill in the `# TODO`s, then run.


In [ ]:
# TODO 1: add VNI 'VNI 50200' (tenant='Tenant-B'), RIDES_ON edge -> 'leaf-07'
# TODO 2: add Service 'Tenant-B App' (layer='VXLAN', criticality='medium'),
#         RUNS_ON edge -> 'VNI 50200'

# G.add_node(...); G.add_edge(...)
# G.add_node(...); G.add_edge(...)

print('At risk, grouped by layer:')
by_layer = {}
for layer, svc, _ in impact(G):
    by_layer.setdefault(layer, []).append(svc)
for layer in sorted(by_layer):
    print(f'  {layer:5}: {by_layer[layer]}')

<details><summary><b>Solution — click to reveal</b></summary>

```python
G.add_node('VNI 50200', label='VNI', tenant='Tenant-B')
G.add_edge('VNI 50200', 'leaf-07', rel='RIDES_ON')

G.add_node('Tenant-B App', label='Service', layer='VXLAN', criticality='medium')
G.add_edge('Tenant-B App', 'VNI 50200', rel='RUNS_ON')

by_layer = {}
for layer, svc, _ in impact(G):
    by_layer.setdefault(layer, []).append(svc)
for layer in sorted(by_layer):
    print(f'{layer}: {by_layer[layer]}')
```

The VXLAN layer now lists **two** tenants, and the total blast radius is five services
across three protocols — all from the same untouched `core-link`. You extended the
cascade onto a new service by adding facts, never logic.
</details>


In [ ]:
# (Solution, applied.)
G.add_node('VNI 50200', label='VNI', tenant='Tenant-B')
G.add_edge('VNI 50200', 'leaf-07', rel='RIDES_ON')
G.add_node('Tenant-B App', label='Service', layer='VXLAN', criticality='medium')
G.add_edge('Tenant-B App', 'VNI 50200', rel='RUNS_ON')
print('Total blast radius now:', sorted((l, s) for l, s, _ in impact(G)))

## Make it yours — put *your* service on the underlay

Now the moment it becomes your tool, not mine. Pick **one** service *you* run and the
**one** protocol layer it rides, then attach it to the shared `igp-reach` through the
matching cross-layer edge. Change the values below and run — your own service name
comes back in the blast radius of the same core link, proving the model understands
*your* network, not just the demo's.

Keep it to the smallest slice that answers one real question. You can always add one
more node tomorrow — that is the whole promise of a graph you can grow.


In [ ]:
# --- change these to your network ---
MY_SERVICE = 'Wiki Portal'          # a service you run
MY_LAYER   = 'BGP'                  # one of: 'BGP', 'MPLS', 'VXLAN'
MY_ANCHOR  = 'my-anchor'           # its next-hop / LSP / leaf handle
# ------------------------------------

# each layer attaches to the shared underlay through its own cross-layer edge
CROSS = {'BGP': 'RESOLVED_BY', 'MPLS': 'BUILT_OVER', 'VXLAN': 'UNDERLAY_VIA'}
ANCHOR_LABEL = {'BGP': 'NextHop', 'MPLS': 'MPLSLSP', 'VXLAN': 'Leaf'}

G.add_node(MY_ANCHOR,  label=ANCHOR_LABEL[MY_LAYER])
G.add_node(MY_SERVICE, label='Service', layer=MY_LAYER, criticality='critical')
G.add_edge(MY_ANCHOR,  'igp-reach', rel=CROSS[MY_LAYER])   # your cross-layer edge
G.add_edge(MY_SERVICE, MY_ANCHOR,  rel='RUNS_ON')

print(f'If the core link fails, is {MY_SERVICE!r} at risk?')
at_risk = [s for _, s, _ in impact(G)]
print('  ->', 'YES' if MY_SERVICE in at_risk else 'no', '  |  full list:', sorted(at_risk))

**Checkpoint.** Your own service name — `Wiki Portal`, or whatever you typed — prints
with `YES` and appears in the full list, right beside the demo's services. That is the
moment the capstone stopped being a tutorial and became yours. Every other service you
run is four lines away, on whichever layer it rides.


## The one rule that keeps a cross-protocol graph from becoming a swamp

A graph that spans four protocols is *more* tempting to over-fill, not less. Hold the
same discipline you learned in every earlier lab:

> **Model the nouns of your design review, not the numbers of your monitoring
> dashboard.**

Routers, links, IGP reachability, next-hops, LSPs, VRFs, leaves, VNIs, services — the
**nouns** you'd draw on a whiteboard while arguing about a design — belong in the graph,
together with the handful of **cross-layer edges** that connect them. Label tables, LFIB
entries, per-flow counters, the 40,000-row LSDB — those are the **numbers**; leave them
in the systems that store them well, and let the graph *reference* them. The graph holds
the *shape of the dependency across layers*; your NMS holds the firehose. Keep that line
sharp and one graph can safely model your whole stack for years.


## You built the capstone

Look back at the distance. You started with an empty page and stacked four protocol
layers onto one shared underlay: an OSPF core link, a BGP next-hop `RESOLVED_BY` it, an
MPLS LSP `BUILT_OVER` it, a VXLAN leaf reaching the fabric `UNDERLAY_VIA` it. Then you
failed **one** link and a single query returned services across **BGP, MPLS, and
VXLAN** — and showed the chain from each one down to the cable. You toggled the link and
the whole cross-protocol blast radius answered as one.

The important idea was never any single protocol. It is this: **operational truth is
layered, and the layers share a floor.** Once those layers and their cross-layer
dependencies are a graph, *"what does this one failure actually take down?"* stops being
four separate guesses in four separate war rooms and becomes one traversal.

**Where to go next:**
- Revisit the four single-protocol labs — OSPF, BGP, MPLS, VXLAN — and notice each was a
  slice of this same graph.
- Run this in the **companion Neo4j lab**, where the two Cypher peeks become queries you
  execute against the full design dataset.
- Add a **change layer**: a `ChangeEvent` node linked to what it touched, and *"what
  changed right before all four services broke?"* becomes one more traversal.

You ran four graphs, one per protocol. Now you can build the one graph that holds them
all — and answer the question none of them could answer alone.
